In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from plotting import *
setup_plotting_style()
print(f"saving figures:{SAVE_FIGURES}")

In [ ]:
import numpy as np
from scipy import signal

SAMPLES_PER_DATASET = 2  # How many rows to visualize per file
SAMPLE_INDICES = [0, 50] # Which rows to pick (adjust if dataset is smaller)

NB_ID = "11"


In [ ]:
def plot_diagnostic_grid(data_row, signal_name ,sample_idx):
    """
    Generates a 5-panel diagnostic plot for a single signal sample.
    """
    fig = plt.figure(figsize=(15, 10))
    fig.suptitle(f"Diagnostic: {signal_name} | Row {sample_idx}", fontsize=16)
    
    # Grid Layout: 2x3 (we will combine some slots)
    gs = fig.add_gridspec(2, 3)

    # Time Domain (Zoomed) - Top Left
    ax_time = fig.add_subplot(gs[0, 0])
    limit = 500 # Zoom in on first 500 samples
    ax_time.plot(data_row.real[:limit], label='I (Real)', alpha=0.8)
    ax_time.plot(data_row.imag[:limit], label='Q (Imag)', alpha=0.8)
    ax_time.set_title("Time Domain (First 500 samples)")
    ax_time.set_xlabel("Sample Index")
    ax_time.legend(loc='upper right')
    ax_time.grid(True, alpha=0.3)

    # Spectrogram - Top Middle & Right (Spanning 2 cols)
    ax_spec = fig.add_subplot(gs[0, 1:])
    f, t, Sxx = signal.spectrogram(data_row, fs=MIT_SAMPLE_RATE, nperseg=256, return_onesided=False)
    # Shift so 0 Hz is in center
    f_shifted = np.fft.fftshift(f)
    Sxx_shifted = np.fft.fftshift(Sxx, axes=0)
    
    im = ax_spec.pcolormesh(t*1000, f_shifted/1e6, 10*np.log10(Sxx_shifted + 1e-12), shading='gouraud', cmap='inferno')
    ax_spec.set_title("Spectrogram")
    ax_spec.set_ylabel("Frequency (MHz)")
    ax_spec.set_xlabel("Time (ms)")
    fig.colorbar(im, ax=ax_spec, label='Power (dB)')

    # PSD (Frequency Domain) - Bottom Left
    ax_psd = fig.add_subplot(gs[1, 0])
    ax_psd.psd(data_row, NFFT=1024, Fs=MIT_SAMPLE_RATE/1e6, color='purple')
    ax_psd.set_title("Power Spectral Density (PSD)")
    ax_psd.set_xlabel("Frequency (MHz)")

    # Constellation (I/Q) - Bottom Middle
    ax_const = fig.add_subplot(gs[1, 1])
    # Downsample for scatter plot clarity
    display_samples = data_row[:5000] 
    ax_const.scatter(display_samples.real, display_samples.imag, s=1, alpha=0.3, color='darkblue')
    ax_const.set_title("Constellation (I vs Q)")
    ax_const.set_xlabel("In-Phase")
    ax_const.set_ylabel("Quadrature")
    ax_const.axis('equal')
    ax_const.grid(True, alpha=0.3)

    # Amplitude Histogram - Bottom Right
    ax_hist = fig.add_subplot(gs[1, 2])
    magnitude = np.abs(data_row)
    ax_hist.hist(magnitude, bins=50, color='green', alpha=0.7, density=True)
    ax_hist.set_title("Amplitude Distribution")
    ax_hist.set_xlabel("Magnitude |x|")
    ax_hist.set_ylabel("Density")
    ax_hist.grid(True, alpha=0.3)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Make room for suptitle

    save_plot(f"diagnostic_raw_{signal_name}_row{sample_idx}", machine_id="B", nb_id=NB_ID, fig_id="01")
    plt.show()

In [ ]:
for sig in SIGNALS:
    file_path = get_unet_path("01_raw", signal=sig)
    
    if not file_path.exists():
        print(f"Skipping {sig} (Not found)")
        continue
        
    print(f"--- Analyzing {sig} ---")
    
    # Load the full dataset
    # Note: We are eager loading here as discussed.
    full_data = np.load(file_path)
    print(f"Loaded shape: {full_data.shape}")
    
    # Process the selected samples
    for idx in SAMPLE_INDICES:
        # Check if index exists in this file
        if idx < full_data.shape[0]:
            print(f"   > Visualizing Row {idx}...")
            row_data = full_data[idx]
            plot_diagnostic_grid(row_data, sig, idx)
        else:
            print(f"   > Index {idx} out of bounds for {sig}")